# Etapa 2 — Comparação de Modelos

Tech Challenge Fase 1 · Pós Tech ML Engineering (FIAP)

Treinamos 3 candidatos com o MESMO split e seed (comparação justa):
Regressão Logística (baseline), Random Forest e MLPClassifier — todos do
Scikit-Learn. A lógica de treino vive em `churn_prediction.train`, reaproveitada
aqui para visualização e discussão em grupo.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from churn_prediction.train import load_data, train_and_compare, select_champion

pd.set_option("display.max_columns", None)

## Treinar os 3 candidatos e comparar

`train_and_compare` já faz o split estratificado com seed fixa (42) e treina
os três modelos nesse MESMO split — é isso que torna a comparação justa.

In [ ]:
df = load_data()
comparison_df, fitted_pipelines = train_and_compare(df)
comparison_df.round(4)

## Discussão em grupo

Antes de rodar a célula do campeão, parem e discutam:
- Qual modelo teve o melhor ROC-AUC? Faz sentido com o que vocês esperavam?
- O Recall da classe Churn variou muito entre os modelos? Por quê isso importa
  pro negócio (ver bloco Impact Simulation do ML Canvas)?
- Algum modelo precisa de ajuste (ex.: threshold, tratamento de desbalanceamento)
  antes de ser considerado "candidato justo"? `MLPClassifier` não tem parâmetro
  `class_weight` nativo — vale a pena investigar se isso penaliza ele
  injustamente na comparação.

In [ ]:
champion_name = select_champion(comparison_df)
print(f"Modelo campeão selecionado por ROC-AUC: {champion_name}")

## Análise de custo de negócio (Impact Simulation do ML Canvas)

Retomando o ML Canvas: custo = (falsos negativos × custo de perder cliente)
+ (falsos positivos × custo de oferta de retenção desperdiçada). Testem abaixo
com os valores que o grupo definiu (ou os valores ilustrativos do Canvas).

In [ ]:
COST_FALSE_NEGATIVE = 1200  # custo de perder um cliente (LTV) -- ajustar com o grupo
COST_FALSE_POSITIVE = 50    # custo de uma oferta de retenção desperdiçada

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from churn_prediction.preprocessing import clean_raw_dataframe, split_features_target
from churn_prediction.train import RANDOM_STATE

df_clean = clean_raw_dataframe(df)
X, y = split_features_target(df_clean)
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

for name, pipeline in fitted_pipelines.items():
    y_pred = pipeline.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    total_cost = fn * COST_FALSE_NEGATIVE + fp * COST_FALSE_POSITIVE
    print(f"{name:22s} | FN={fn:4d} FP={fp:4d} | custo total = R$ {total_cost:,.2f}")

**Importante**: o modelo com melhor ROC-AUC não é necessariamente o de menor
custo de negócio — comparem os dois critérios antes de fechar a escolha final
com o grupo.

## Salvar o campeão

Rodem via terminal (não aqui no notebook), para deixar o processo reprodutível
e testável por qualquer membro do grupo:

```bash
python -m churn_prediction.train
```

Isso salva `models/champion_model.joblib` e `models/model_comparison.csv`.

## Próximos passos (Etapa 3)

- Rodar os testes (`pytest`) para confirmar que a API reconhece o novo campeão.
- Subir a API (`uvicorn churn_prediction.api.main:app --reload`) e testar o
  `/predict` manualmente.
- Revisar se algum ajuste de pré-processamento é necessário para o modelo
  campeão escolhido.